In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, roc_auc_score

In [2]:
df = pd.read_csv('cleaned_churn_data.csv')

In [17]:
y = df['Churn']
X = df.drop('Churn', axis=1)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = ['Contract','InternetService','PaymentMethod']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(), cat_cols)
],
remainder='passthrough')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=4))
])

In [18]:
tree_pipeline.fit(X_train, y_train)

c:\Users\v-acherlapal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])])),
                ('model', DecisionTreeClassifier(max_depth=4))])

In [19]:
y_pred = tree_pipeline.predict(X_test)
y_prob = tree_pipeline.predict_proba(X_test)[:, 1]

In [20]:
train_pred = tree_pipeline.predict(X_train)
training_accuracy = accuracy_score(y_train, train_pred)
print("Training Accuracy:", training_accuracy)

Training Accuracy: 0.7944888888888889


In [21]:
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.767590618336887


In [22]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[843 190]
 [137 237]]


In [23]:
precision_tree = precision_score(y_test, y_pred)
recall_tree = recall_score(y_test, y_pred)
roc_auc_tree = roc_auc_score(y_test, y_prob)

print("Precision:", precision_tree)
print("Recall:", recall_tree)
print("ROC-AUC:", roc_auc_tree)

Precision: 0.5550351288056206
Recall: 0.6336898395721925
ROC-AUC: 0.8174454240025677


In [24]:
from sklearn.ensemble import RandomForestClassifier

In [25]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_estimators=200))
])

In [26]:
rf_pipeline.fit(X_train, y_train)

c:\Users\v-acherlapal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [31]:
rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:,1]

In [28]:
test_accuracy = accuracy_score(y_test, rf_pred)
print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.7810945273631841


In [29]:
rf_train_pred = rf_pipeline.predict(X_train)
rf_train_accuracy = accuracy_score(y_train, rf_train_pred)
print("Train Accuracy:", rf_train_accuracy)

Train Accuracy: 0.9976888888888888


In [32]:
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))

Accuracy: 0.7810945273631841
Precision: 0.609271523178808
Recall: 0.4919786096256685
ROC-AUC: 0.8105603325550937
Confusion Matrix:
 [[915 118]
 [190 184]]


In [33]:

importances = rf_pipeline.named_steps["model"].feature_importances_
features = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(importance_df.head(10))

                                Feature  Importance
2                     num__TotalCharges    0.226003
1                   num__MonthlyCharges    0.214803
0                           num__tenure    0.178084
3          cat__Contract_Month-to-month    0.068259
7      cat__InternetService_Fiber optic    0.042091
11  cat__PaymentMethod_Electronic check    0.031365
13                    remainder__gender    0.026935
25          remainder__PaperlessBilling    0.026356
5                cat__Contract_Two year    0.025073
15                   remainder__Partner    0.023451


Random Forest mainly improves variance, not necessarily recall.\
The algorithm is excellent for stable predictions, but it sometimes becomes slightly conservative.

In [34]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])

In [35]:
gb_pipeline.fit(X_train, y_train)

c:\Users\v-acherlapal\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])])),
                ('model', GradientBoostingClassifier(random_state=42))])

In [36]:
y_pred_gb = gb_pipeline.predict(X_test)
y_prob_gb = gb_pipeline.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Precision:", precision_score(y_test, y_pred_gb))
print("Recall:", recall_score(y_test, y_pred_gb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))

Accuracy: 0.7874911158493249
Precision: 0.6288659793814433
Recall: 0.4893048128342246
ROC-AUC: 0.8340938857281891
Confusion Matrix:
 [[925 108]
 [191 183]]


In [38]:
train_pred_gb = gb_pipeline.predict(X_train)
train_accuracy_gb = accuracy_score(y_train, train_pred_gb)
print("Training Accuracy:", train_accuracy_gb)

Training Accuracy: 0.8264888888888889
